# 01 - Análise Exploratória de Dados (EDA) & Baseline

Notebook para exploração inicial do dataset de telecomunicações e criação de um modelo baseline para previsão de Churn.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, precision_score,
    recall_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay
)
import mlflow

from src.data_loader import load_telco_churn

SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

%matplotlib inline

## 1. Carregamento dos Dados

Utilizamos a função `load_data` do módulo `src.data_loader` para carregar o dataset **Telco Customer Churn**.
Este dataset contém informações de clientes de uma empresa de telecomunicações, incluindo dados demográficos, serviços contratados e se o cliente cancelou (Churn = Yes) ou não (Churn = No).

In [ ]:
df = load_telco_churn()
df.head()

In [ ]:
df.info()

## 2. Limpeza da coluna `TotalCharges`

Esta é uma **pegadinha clássica** deste dataset: a coluna `TotalCharges` é importada com dtype `object` (string) em vez de `float64`.

**Causa raiz:** existem registros com espaços em branco (`" "`) no lugar de valores numéricos. Esses registros pertencem a clientes novos com `tenure = 0`, ou seja, clientes que acabaram de entrar e ainda não tiveram nenhuma cobrança total registrada.

**Estratégia de tratamento:**
1. Converter a coluna para numérico com `pd.to_numeric(..., errors='coerce')` — valores inválidos viram `NaN`.
2. Remover as linhas com `NaN` usando `dropna()`. São apenas ~11 registros em ~7.000, um impacto desprezível.

In [ ]:
# Converte TotalCharges para numérico (espaços em branco viram NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"Valores nulos em TotalCharges: {df['TotalCharges'].isna().sum()}")
print(f"Total de linhas antes da limpeza: {len(df)}")

# Remove as linhas com valores nulos
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Total de linhas após limpeza: {len(df)}")
print(f"Nulos restantes: {df.isnull().sum().sum()}")

## 2.1 Visão Geral do Dataset

Resumo estatístico das features numéricas e verificação de valores únicos por coluna.

## 3. Distribuição da Variável Alvo (`Churn`)

Antes de qualquer modelagem, é fundamental entender o **balanceamento da variável alvo**.

Datasets de Churn são tipicamente **desbalanceados** — a maioria dos clientes permanece ativa e apenas uma fração cancela. Esse desbalanceamento impacta diretamente a escolha de métricas (acurácia pode ser enganosa) e pode exigir técnicas como oversampling, undersampling ou ajuste de pesos na função de perda da rede neural.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='Churn', ax=ax, palette='Set2')
ax.set_title('Distribuição da Variável Alvo (Churn)')
ax.set_xlabel('Churn')
ax.set_ylabel('Quantidade de Clientes')

# Mostra as contagens e percentuais
total = len(df)
for p in ax.patches:
    count = int(p.get_height())
    pct = f'{100 * count / total:.1f}%'
    ax.annotate(f'{count}\n({pct})', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas\n")
print("Valores únicos por coluna:")
print(df.nunique().to_string())
print(f"\nValores nulos restantes: {df.isnull().sum().sum()}")
df.describe()

In [ ]:
df.drop(columns=['customerID'], inplace=True)

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('Churn')

print(f"Features numéricas ({len(num_cols)}): {num_cols}")
print(f"Features categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Variável alvo: Churn")

## 4. Análise de Features Numéricas

Distribuição de `tenure`, `MonthlyCharges` e `TotalCharges`, segmentadas por Churn.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, num_cols):
    sns.histplot(data=df, x=col, hue='Churn', kde=True, ax=ax, palette='Set2')
    ax.set_title(f'Distribuição de {col} por Churn')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df, x='Churn', y=col, ax=ax, palette='Set2')
    ax.set_title(f'{col} por Churn')

plt.tight_layout()
plt.show()

## 5. Análise de Features Categóricas

Taxa de churn por categoria para cada feature categórica. Isso revela quais categorias têm maior propensão ao cancelamento.

In [ ]:
n_cols_grid = 4
n_rows_grid = (len(cat_cols) + n_cols_grid - 1) // n_cols_grid

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(20, 4 * n_rows_grid))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean()).sort_values(ascending=False)
    churn_rate.plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2'), edgecolor='black')
    axes[i].set_title(f'Taxa de Churn por {col}')
    axes[i].set_ylabel('Taxa de Churn')
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=45)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 6. Matriz de Correlação

Correlação entre as features numéricas e a variável alvo (codificada como 0/1).

In [ ]:
df_corr = df.copy()
df_corr['Churn_num'] = (df_corr['Churn'] == 'Yes').astype(int)

corr_cols = num_cols + ['Churn_num']
corr_matrix = df_corr[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Matriz de Correlação (Features Numéricas + Churn)')
plt.tight_layout()
plt.show()

## 7. Data Readiness — Resumo da EDA

**Volume:** ~7.000 registros, 20 features — volume adequado para MLP tabular.

**Qualidade:** Apenas ~11 registros com dados faltantes (TotalCharges), já removidos. Sem duplicatas significativas.

**Desbalanceamento:** A classe minoritária (Churn=Yes) representa ~26% do dataset. É um desbalanceamento moderado — exige atenção na escolha de métricas e possível ajuste de pesos na loss function.

**Insights principais:**
- **tenure** é o preditor numérico mais forte: clientes com menor tempo de contrato têm muito mais churn.
- **MonthlyCharges** alto correlaciona com maior churn.
- **Contract** mensal tem taxa de churn muito superior a contratos de 1–2 anos.
- **InternetService** via fibra óptica tem churn significativamente maior que DSL.
- Features de serviços adicionais (OnlineSecurity, TechSupport) atuam como fatores protetivos.

**Métricas escolhidas:**
- **Técnicas:** AUC-ROC (ranking geral), PR-AUC (foco na classe positiva), F1-Score, Recall.
- **Negócio:** Custo de churn evitado — falso negativo (não identificar churn) é mais custoso que falso positivo (oferecer retenção desnecessária).

---

## 8. Pré-processamento

Pipeline de pré-processamento com `ColumnTransformer`:
- **Numéricas** (`tenure`, `MonthlyCharges`, `TotalCharges`): `StandardScaler`
- **Categóricas**: `OneHotEncoder` com `drop='if_binary'` para evitar multicolinearidade

Divisão estratificada 80/20 para manter a proporção de Churn em treino e teste.

In [ ]:
X = df.drop(columns=['Churn'])
y = (df['Churn'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print(f"Proporção Churn (treino): {y_train.mean():.2%}")
print(f"Proporção Churn (teste):  {y_test.mean():.2%}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False), cat_cols),
    ]
)

---

## 9. Baselines com MLflow Tracking

Treinamos dois modelos baseline para estabelecer o piso de performance:

1. **DummyClassifier** (`strategy='most_frequent'`): sempre prediz a classe majoritária (No Churn). Serve como lower bound.
2. **Regressão Logística**: modelo linear clássico, rápido e interpretável. Serve como baseline competitivo.

Todos os experimentos são registrados no MLflow com parâmetros, métricas e artefatos.

In [ ]:
mlflow.set_experiment("churn-prediction")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate_model(model, X_test, y_test):
    """Calcula métricas de avaliação para um modelo treinado."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred.astype(float)

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred),
        'auc_roc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
    }
    return metrics, y_pred, y_proba

### 9.1 DummyClassifier (Lower Bound)

In [ ]:
dummy_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent', random_state=SEED)),
])

with mlflow.start_run(run_name="dummy-classifier"):
    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("seed", SEED)
    mlflow.set_tag("stage", "baseline")

    dummy_pipe.fit(X_train, y_train)
    dummy_metrics, dummy_pred, dummy_proba = evaluate_model(dummy_pipe, X_test, y_test)

    for name, value in dummy_metrics.items():
        mlflow.log_metric(name, value)

    print("DummyClassifier — Métricas no conjunto de teste:")
    for name, value in dummy_metrics.items():
        print(f"  {name}: {value:.4f}")

### 9.2 Regressão Logística

In [ ]:
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')),
])

with mlflow.start_run(run_name="logistic-regression"):
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("seed", SEED)
    mlflow.set_tag("stage", "baseline")

    # Validação cruzada estratificada
    cv_results = cross_validate(
        lr_pipe, X_train, y_train, cv=cv,
        scoring=['accuracy', 'f1', 'roc_auc'],
        return_train_score=False
    )

    print("Validação Cruzada (5-fold) — Regressão Logística:")
    for metric in ['accuracy', 'f1', 'roc_auc']:
        scores = cv_results[f'test_{metric}']
        print(f"  {metric}: {scores.mean():.4f} ± {scores.std():.4f}")
        mlflow.log_metric(f"cv_{metric}_mean", scores.mean())
        mlflow.log_metric(f"cv_{metric}_std", scores.std())

    # Treina no conjunto completo de treino e avalia no teste
    lr_pipe.fit(X_train, y_train)
    lr_metrics, lr_pred, lr_proba = evaluate_model(lr_pipe, X_test, y_test)

    print("\nMétricas no conjunto de teste:")
    for name, value in lr_metrics.items():
        mlflow.log_metric(name, value)
        print(f"  {name}: {value:.4f}")

## 10. Comparação dos Baselines

Tabela comparativa e visualizações de performance.

In [ ]:
comparison = pd.DataFrame({
    'DummyClassifier': dummy_metrics,
    'Logistic Regression': lr_metrics,
}).T

comparison.style.format('{:.4f}').highlight_max(axis=0, color='lightgreen')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Matriz de confusão — Regressão Logística
ConfusionMatrixDisplay.from_predictions(y_test, lr_pred, ax=axes[0], cmap='Blues')
axes[0].set_title('Matriz de Confusão — Logistic Regression')

# Curva ROC
RocCurveDisplay.from_predictions(y_test, lr_proba, ax=axes[1], name='Logistic Regression')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.50)')
axes[1].set_title('Curva ROC')
axes[1].legend()

# Curva Precision-Recall
PrecisionRecallDisplay.from_predictions(y_test, lr_proba, ax=axes[2], name='Logistic Regression')
axes[2].set_title('Curva Precision-Recall')

plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, lr_pred, target_names=['No Churn', 'Churn']))

## 11. Análise de Custo — Trade-off FP vs. FN

No contexto de churn, os erros têm custos assimétricos:
- **Falso Negativo (FN):** cliente cancela sem ser identificado → perda de receita recorrente.
- **Falso Positivo (FP):** cliente recebe oferta de retenção desnecessária → custo menor (desconto/promoção).

Portanto, **Recall** (capturar o máximo de churners) é mais importante que **Precision** neste cenário.

In [ ]:
CUSTO_FN = 500  # Perda de receita por cliente não identificado (churn real)
CUSTO_FP = 50   # Custo de oferta de retenção para cliente que não iria cancelar

cm = confusion_matrix(y_test, lr_pred)
tn, fp, fn, tp = cm.ravel()

custo_total = (fn * CUSTO_FN) + (fp * CUSTO_FP)
custo_sem_modelo = y_test.sum() * CUSTO_FN  # Se não fizéssemos nada, todos os churners seriam FN
economia = custo_sem_modelo - custo_total

print(f"Matriz de Confusão: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"\nCusto por FN (churn não detectado): R$ {CUSTO_FN}")
print(f"Custo por FP (retenção desnecessária): R$ {CUSTO_FP}")
print(f"\nCusto total dos erros (com modelo): R$ {custo_total:,.0f}")
print(f"Custo sem modelo (todos FN):          R$ {custo_sem_modelo:,.0f}")
print(f"Economia gerada pelo modelo:          R$ {economia:,.0f}")